# Local Text Agent

In [ ]:
from agents.text import TextAgent, TextModelConfig
vllm_config = TextModelConfig(
    backend="vllm",
    model_name="google/gemma-3-1b-it",
    model_path="google/gemma-3-1b-it",
    device="mps",
    torch_dtype="float16",
    max_new_tokens=256,
    temperature=0.7,
    llamacpp_params={
        "n_gpu_layers": 1,
        "tensor_parallel_size": 1,  # Явное указание для MPS
        "mps": True
    },
    disable_distributed=True,
    quantized=False  # Отключаем квантование для MPS
)

agent = TextAgent(
    config=vllm_config,
    title_params={
        "temperature": 0.3,
        "max_new_tokens": 30,
        "top_p": 0.9,
        "repetition_penalty": 1.2,
        "stop_sequences": ["\n"]
    }
)

In [ ]:
# Генерация заголовка
title = agent.generate_title("Future of AI in healthcare")
print(f"Generated Title: {title}")
# Output: "AI-Driven Innovations Transforming Healthcare Delivery"

# Генерация контента
content = agent.generate_content(
    "Technical explanation of neural networks",
    format_hint="bullet_list"
)
print("Generated Content:")
print(content)
"""
- Neural networks are computational models inspired by biological neurons
- Consist of interconnected layers (input, hidden, output)
- Use activation functions like ReLU and Sigmoid for non-linear transformations
- Trained via backpropagation and gradient descent optimization
"""

# API Text Agent

In [ ]:
from agents.text import TextAgent, TextModelConfig

import requests


yandex_config = TextModelConfig(
    backend="api",
    api_base="https://llm.api.cloud.yandex.net/llm/v1alpha/instruct",
    api_key="your_iam_token",
    model_name="yandexgpt-lite"
)


agent = TextAgent(yandex_config)

In [ ]:
def generate_yandex_gpt(prompt: str) -> str:
    headers = {
        "Authorization": f"Bearer {agent.config.api_key}",
        "Content-Type": "application/json"
    }
    
    data = {
        "model": agent.config.model_name,
        "messages": [{
            "role": "user",
            "content": prompt
        }],
        "temperature": 0.7,
        "max_tokens": 300
    }
    
    response = requests.post(
        agent.config.api_base,
        headers=headers,
        json=data
    )
    return response.json()["result"]["alternatives"][0]["message"]["text"]

# Использование
title = generate_yandex_gpt("Generate title about AI in healthcare")
content = generate_yandex_gpt("Explain neural networks for beginners")